# Kratos MCP Server — multi-stage orchestration & Flowgraph interop

This notebook drives the **real** `kratos-mcp` server over stdio to show three
capabilities that go beyond a single simulation:

- **`create_multistage_project`** — chain several analyses into one *orchestrated*
  case (Kratos' `SequentialOrchestrator`), here two load steps on a shared mesh;
- **`explain_project_parameters`** — get a structured summary of any case;
- **Flowgraph interop** — convert a case to a
  [Kratos FlowGraph](https://github.com/KratosMultiphysics/Flowgraph) node graph and
  back, losslessly.

Pairs with the `kratos://examples/multistage-load-steps` resource and the
`docs/tutorials/multistage.md` tutorial. Every output is from a live run against
a compiled Kratos 10.4 build.

In [1]:
import asyncio
import base64
import json
import shutil
import sys
import tempfile
from pathlib import Path
from contextlib import AsyncExitStack

from IPython.display import Image as IPImage, Markdown, display

from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client


def _preview(value, limit=4000):
    text = value if isinstance(value, str) else json.dumps(value, indent=2, default=str)
    if len(text) > limit:
        text = text[: limit] + f"\n... [truncated, {len(text)} chars total]"
    print(text)


async def call(tool_name, **kwargs):
    # Call an MCP tool, print its JSON result, display any inline images,
    # and return the parsed value for use in later cells.
    result = await session.call_tool(tool_name, kwargs)
    value = None
    for block in result.content:
        if block.type == "text":
            try:
                value = json.loads(block.text)
            except json.JSONDecodeError:
                value = block.text
        elif block.type == "image":
            display(IPImage(data=base64.b64decode(block.data), format=block.mimeType.split("/")[-1]))
    _preview(value)
    return value


async def read_resource(uri):
    result = await session.read_resource(uri)
    return result.contents[0].text


async def get_prompt(name, **kwargs):
    result = await session.get_prompt(name, kwargs)
    return result.messages[0].content.text

## Connect to the server\n\nThe server's stdio pipes are managed by anyio context managers whose cancel scopes are tied to the asyncio Task that opened them. Every Jupyter cell runs its top-level `await` in a *new* Task, so we run the whole connect → disconnect lifecycle inside one persistent background task kept alive on an `asyncio.Event`, and only *signal* it to close.

In [2]:
session = None
_stop_event = asyncio.Event()
_ready_event = asyncio.Event()


async def _server_lifecycle():
    global session
    async with AsyncExitStack() as stack:
        params = StdioServerParameters(command=sys.executable, args=["-m", "kratos_mcp.server"])
        read, write = await stack.enter_async_context(stdio_client(params))
        session = await stack.enter_async_context(ClientSession(read, write))
        await session.initialize()
        _ready_event.set()
        await _stop_event.wait()  # keeps this task (and the session) alive


_server_task = asyncio.create_task(_server_lifecycle())
await _ready_event.wait()

tools = await session.list_tools()
resources = await session.list_resources()
prompts = await session.list_prompts()
print(f"{len(tools.tools)} tools, {len(resources.resources)} resources, "
      f"{len(prompts.prompts)} prompts")
print("resources:", [str(r.uri) for r in resources.resources])

40 tools, 13 resources, 5 prompts
resources: ['kratos://docs/mdpa-format', 'kratos://docs/project-parameters', 'kratos://docs/materials', 'kratos://examples/cantilever', 'kratos://examples/thermal-bar', 'kratos://examples/naca-airfoil', 'kratos://examples/lid-driven-cavity', 'kratos://examples/plasticity-cube', 'kratos://examples/multistage-load-steps', 'kratos://examples/channel-flow', 'kratos://examples/modal-box', 'kratos://examples/dynamic-cantilever', 'kratos://examples/potential-flow']


## 1. What multi-stage cases look like

A multi-stage case chains analyses with an `orchestrator` + `stages` + `execution_list` structure instead of a single solver. Read the worked example to see the shape and its verified result:

In [3]:
print(await read_resource("kratos://examples/multistage-load-steps"))

# Worked example: multi-stage load steps (orchestrated analysis)

A cantilever solved in TWO sequential stages that share one mesh, driven by
Kratos' SequentialOrchestrator -- the shape create_multistage_project
produces. Stage `load_step_1` applies a 1 MN/m downward line load on the
right edge; stage `load_step_2` reuses the same model part
(input_type "use_input_model_part") and applies 2 MN/m. This is the
`orchestrator` + `stages` + `execution_list` ProjectParameters structure, run
by run_simulation exactly like a single-stage case.

Real files on disk (src/kratos_mcp/examples/multistage_load_steps/); the mesh
is a 10x4 rectangle cantilever (55 nodes) from mdpa_create_structured_mesh.

## mesh.mdpa

```
Begin ModelPartData
End ModelPartData

Begin Properties 1
End Properties

Begin Nodes
    1 0 0 0
    2 0.1 0 0
    3 0.2 0 0
    4 0.3 0 0
    5 0.4 0 0
    6 0.5 0 0
    7 0.6 0 0
    8 0.7 0 0
    9 0.8 0 0
    10 0.9 0 0
    11 1 0 0
    12 0 0.05 0
    13 0.1 0.05 0
    14 0.2 0

## 2. Compose the orchestrated case

`create_multistage_project` builds the whole thing from ordinary single-stage templates: two `structural_static` stages that share one mesh (the second reuses the first's model part via `use_input_model_part`).

In [4]:
workdir = Path(tempfile.mkdtemp(prefix="kratos-mcp-multistage-"))
await call("create_multistage_project", directory=str(workdir),
           name="cantilever_ms",
           stages=[{"name": "load_step_1", "template": "structural_static"},
                   {"name": "load_step_2", "template": "structural_static"}])

{
  "case_dir": "/tmp/kratos-mcp-multistage-en_d2ngz",
  "created": [
    "/tmp/kratos-mcp-multistage-en_d2ngz/ProjectParameters.json",
    "/tmp/kratos-mcp-multistage-en_d2ngz/Materials.json"
  ],
  "execution_list": [
    "load_step_1",
    "load_step_2"
  ],
  "next_steps": [
    "Create the shared mesh at /tmp/kratos-mcp-multistage-en_d2ngz/mesh.mdpa (e.g. with mdpa_create_structured_mesh)",
    "Validate with validate_case('/tmp/kratos-mcp-multistage-en_d2ngz')",
    "Run with run_simulation('/tmp/kratos-mcp-multistage-en_d2ngz')"
  ]
}


{'case_dir': '/tmp/kratos-mcp-multistage-en_d2ngz',
 'created': ['/tmp/kratos-mcp-multistage-en_d2ngz/ProjectParameters.json',
  '/tmp/kratos-mcp-multistage-en_d2ngz/Materials.json'],
 'execution_list': ['load_step_1', 'load_step_2'],
 'next_steps': ['Create the shared mesh at /tmp/kratos-mcp-multistage-en_d2ngz/mesh.mdpa (e.g. with mdpa_create_structured_mesh)',
  "Validate with validate_case('/tmp/kratos-mcp-multistage-en_d2ngz')",
  "Run with run_simulation('/tmp/kratos-mcp-multistage-en_d2ngz')"]}

## 3. Add the load steps and the shared mesh

Each stage carries its own load inside `stage_settings`; we set an increasing line load per stage, then generate the shared cantilever mesh.

In [5]:
# The load lives INSIDE each stage's stage_settings, so we add it directly to
# the composed file (add_boundary_condition targets single-stage cases). Stage 1
# gets 1 MN/m, stage 2 gets 2 MN/m; we also send each stage's VTK to its own
# folder so we can read both stages' results.
pp = json.loads((workdir / "ProjectParameters.json").read_text())
for name, modulus in (("load_step_1", 1.0e6), ("load_step_2", 2.0e6)):
    ss = pp["stages"][name]["stage_settings"]
    ss.setdefault("processes", {}).setdefault("loads_process_list", []).append({
        "python_module": "assign_vector_by_direction_to_condition_process",
        "kratos_module": "KratosMultiphysics",
        "process_name": "AssignVectorByDirectionToConditionProcess",
        "Parameters": {
            "model_part_name": "Structure.right", "variable_name": "LINE_LOAD",
            "modulus": modulus, "direction": [0.0, -1.0, 0.0], "interval": [0.0, "End"],
        },
    })
    ss["output_processes"]["vtk_output"][0]["Parameters"]["output_path"] = (
        "vtk_" + name)
(workdir / "ProjectParameters.json").write_text(json.dumps(pp, indent=4))
print("injected loads: load_step_1 = 1 MN/m, load_step_2 = 2 MN/m")

injected loads: load_step_1 = 1 MN/m, load_step_2 = 2 MN/m


In [6]:
await call("mdpa_create_structured_mesh", path=str(workdir / "mesh.mdpa"),
           kind="rectangle", size=[1.0, 0.2], divisions=[10, 4])

{
  "written_to": "/tmp/kratos-mcp-multistage-en_d2ngz/mesh.mdpa",
  "num_nodes": 55,
  "num_elements": 40,
  "num_conditions": 28,
  "elements_by_type": {
    "SmallDisplacementElement2D4N": 40
  },
  "conditions_by_type": {
    "LineLoadCondition2D2N": 28
  },
  "properties_ids": [
    1
  ],
  "bounding_box": {
    "min": [
      0.0,
      0.0,
      0.0
    ],
    "max": [
      1.0,
      0.2,
      0.0
    ]
  },
  "sub_model_parts": {
    "domain": {
      "nodes": 55,
      "elements": 40,
      "conditions": 0,
      "sub_model_parts": {}
    },
    "left": {
      "nodes": 5,
      "elements": 0,
      "conditions": 4,
      "sub_model_parts": {}
    },
    "right": {
      "nodes": 5,
      "elements": 0,
      "conditions": 4,
      "sub_model_parts": {}
    },
    "bottom": {
      "nodes": 11,
      "elements": 0,
      "conditions": 10,
      "sub_model_parts": {}
    },
    "top": {
      "nodes": 11,
      "elements": 0,
      "conditions": 10,
      "sub_model_parts"

{'written_to': '/tmp/kratos-mcp-multistage-en_d2ngz/mesh.mdpa',
 'num_nodes': 55,
 'num_elements': 40,
 'num_conditions': 28,
 'elements_by_type': {'SmallDisplacementElement2D4N': 40},
 'conditions_by_type': {'LineLoadCondition2D2N': 28},
 'properties_ids': [1],
 'bounding_box': {'min': [0.0, 0.0, 0.0], 'max': [1.0, 0.2, 0.0]},
 'sub_model_parts': {'domain': {'nodes': 55,
   'elements': 40,
   'conditions': 0,
   'sub_model_parts': {}},
  'left': {'nodes': 5, 'elements': 0, 'conditions': 4, 'sub_model_parts': {}},
  'right': {'nodes': 5, 'elements': 0, 'conditions': 4, 'sub_model_parts': {}},
  'bottom': {'nodes': 11,
   'elements': 0,
   'conditions': 10,
   'sub_model_parts': {}},
  'top': {'nodes': 11,
   'elements': 0,
   'conditions': 10,
   'sub_model_parts': {}}}}

## 4. Validate and run both stages

`validate_case` understands the multi-stage structure; `run_simulation` drives it through the orchestrator. The log shows two analyses.

In [7]:
await call("validate_case", case_dir=str(workdir))

{
  "valid": true,
  "issues": [],
  "warnings": [
    "Multi-stage case: per-stage Kratos-side solver validation is skipped (validated at run time)."
  ]
}


{'valid': True,
 'issues': [],
 'warnings': ['Multi-stage case: per-stage Kratos-side solver validation is skipped (validated at run time).']}

In [8]:
run = await call("run_simulation", case_dir=str(workdir), wait_seconds=120)
job_id = run["job_id"]

{
  "job_id": "20260714-173359-6ae74b",
  "case_dir": "/tmp/kratos-mcp-multistage-en_d2ngz",
  "parameters_file": "ProjectParameters.json",
  "state": "succeeded",
  "pid": 2854450,
  "returncode": 0,
  "created_at": 1784043239.0567358,
  "started_at": 1784043239.0567358,
  "finished_at": 1784043240.859051,
  "analysis_type": null,
  "extra": {},
  "elapsed_seconds": 1.8
}


In [9]:
logs = await call("job_logs", job_id=job_id, tail=250)
text = logs if isinstance(logs, str) else logs.get("log", "")
print("Analysis -END- count:", text.count("Analysis -END-"))

{
  "job_id": "20260714-173359-6ae74b",
  "log": "Maximum number of threads: 20.\nRunning without MPI.\n[WARNING] Current analysis stage 'KratosMultiphysics.StructuralMechanicsApplication.structural_mechanics_analysis' is not registered. Assuming that provided string contains module and class names.: \nImporting    KratosStructuralMechanicsApplication \n    KRATOS   ___|  |                   |                   |\n           \\___ \\  __|  __| |   |  __| __| |   |  __| _` | |\n                 | |   |    |   | (    |   |   | |   (   | |\n           _____/ \\__|_|   \\__,_|\\___|\\__|\\__,_|_|  \\__,_|_| MECHANICS\nInitializing KratosStructuralMechanicsApplication...\nImporting    KratosConstitutiveLawsApplication \nInitializing KratosConstitutiveLawsApplication...\n::[MechanicalSolver]:: : Construction finished \n::[StaticMechanicalSolver]:: : Construction finished \n::[MechanicalSolver]:: : Variables ADDED \n::[PythonSolver]::: Reading model part. \nSingleImportModelPart: Reading mode

## 5. Explain the case

`explain_project_parameters` returns a structured summary of any ProjectParameters — here it recognises the multi-stage structure and summarises each stage.

In [10]:
await call("explain_project_parameters", parameters_file=str(workdir / "ProjectParameters.json"))

{
  "kind": "multi_stage",
  "orchestrator": "SequentialOrchestrator",
  "execution_list": [
    "load_step_1",
    "load_step_2"
  ],
  "num_stages": 2,
  "stages": [
    {
      "name": "load_step_1",
      "analysis_type": "structural",
      "analysis_stage": "KratosMultiphysics.StructuralMechanicsApplication.structural_mechanics_analysis",
      "problem_name": "cantilever_ms_load_step_1",
      "parallel_type": "OpenMP",
      "start_time": 0.0,
      "end_time": 1.0,
      "solver": {
        "solver_type": "Static",
        "model_part_name": "Structure",
        "domain_size": 2
      },
      "linear_solver": {
        "solver_type": "LinearSolversApplication.sparse_lu"
      },
      "model_import": {
        "input_type": "mdpa",
        "input_filename": "mesh"
      },
      "materials": {
        "materials_filename": "Materials.json"
      },
      "processes": [
        {
          "list": "constraints_process_list",
          "python_module": "assign_vector_variable_p

{'kind': 'multi_stage',
 'orchestrator': 'SequentialOrchestrator',
 'execution_list': ['load_step_1', 'load_step_2'],
 'num_stages': 2,
 'stages': [{'name': 'load_step_1',
   'analysis_type': 'structural',
   'analysis_stage': 'KratosMultiphysics.StructuralMechanicsApplication.structural_mechanics_analysis',
   'problem_name': 'cantilever_ms_load_step_1',
   'parallel_type': 'OpenMP',
   'start_time': 0.0,
   'end_time': 1.0,
   'solver': {'solver_type': 'Static',
    'model_part_name': 'Structure',
    'domain_size': 2},
   'linear_solver': {'solver_type': 'LinearSolversApplication.sparse_lu'},
   'model_import': {'input_type': 'mdpa', 'input_filename': 'mesh'},
   'materials': {'materials_filename': 'Materials.json'},
   'processes': [{'list': 'constraints_process_list',
     'python_module': 'assign_vector_variable_process',
     'kratos_module': 'KratosMultiphysics',
     'process_name': 'AssignVectorVariableProcess',
     'model_parts': ['Structure.left']},
    {'list': 'loads_pro

## 6. Round-trip through Flowgraph

Convert the case to a [Kratos FlowGraph](https://github.com/KratosMultiphysics/Flowgraph) node graph (openable in the visual editor), then convert it straight back — the reconstruction reproduces the original exactly.

In [11]:
await call("export_case_to_flowgraph",
           parameters_file=str(workdir / "ProjectParameters.json"),
           output_file=str(workdir / "graph.json"))

{
  "graph": {
    "last_node_id": 17,
    "last_link_id": 16,
    "nodes": [
      {
        "id": 1,
        "type": "Orchestrators/SequentialOrchestrator",
        "pos": [
          780,
          100
        ],
        "size": [
          220,
          100
        ],
        "flags": {},
        "mode": 0,
        "inputs": [],
        "outputs": [],
        "properties": {
          "_role": "orchestrator",
          "_fragment": {
            "name": "Orchestrators.KratosMultiphysics.SequentialOrchestrator",
            "settings": {
              "echo_level": 0,
              "execution_list": [
                "load_step_1",
                "load_step_2"
              ],
              "load_from_checkpoint": null,
              "stage_checkpoints": false
            }
          }
        }
      },
      {
        "id": 2,
        "type": "Solvers/Structural mechanics/StructuralMechanicsSolver",
        "pos": [
          260,
          100
        ],
        "size": [
     

{'graph': {'last_node_id': 17,
  'last_link_id': 16,
  'nodes': [{'id': 1,
    'type': 'Orchestrators/SequentialOrchestrator',
    'pos': [780, 100],
    'size': [220, 100],
    'flags': {},
    'mode': 0,
    'inputs': [],
    'outputs': [],
    'properties': {'_role': 'orchestrator',
     '_fragment': {'name': 'Orchestrators.KratosMultiphysics.SequentialOrchestrator',
      'settings': {'echo_level': 0,
       'execution_list': ['load_step_1', 'load_step_2'],
       'load_from_checkpoint': None,
       'stage_checkpoints': False}}}},
   {'id': 2,
    'type': 'Solvers/Structural mechanics/StructuralMechanicsSolver',
    'pos': [260, 100],
    'size': [220, 100],
    'flags': {},
    'mode': 0,
    'inputs': [],
    'outputs': [],
    'properties': {'_role': 'solver_settings',
     '_analysis_stage': 'KratosMultiphysics.StructuralMechanicsApplication.structural_mechanics_analysis',
     '_process_lists': ['constraints_process_list',
      'loads_process_list',
      'list_other_process

In [12]:
back = await call("import_flowgraph_to_case",
                   graph_file=str(workdir / "graph.json"),
                   output_file=str(workdir / "roundtrip.json"))
original = json.loads((workdir / "ProjectParameters.json").read_text())
print("round-trip reproduces the original exactly:",
      json.dumps(original, sort_keys=True) == json.dumps(back["parameters"], sort_keys=True))

{
  "parameters": {
    "orchestrator": {
      "name": "Orchestrators.KratosMultiphysics.SequentialOrchestrator",
      "settings": {
        "echo_level": 0,
        "execution_list": [
          "load_step_1",
          "load_step_2"
        ],
        "load_from_checkpoint": null,
        "stage_checkpoints": false
      }
    },
    "stages": {
      "load_step_1": {
        "stage_settings": {
          "solver_settings": {
            "solver_type": "Static",
            "model_part_name": "Structure",
            "domain_size": 2,
            "echo_level": 1,
            "analysis_type": "linear",
            "time_stepping": {
              "time_step": 1.1
            },
            "rotation_dofs": false,
            "linear_solver_settings": {
              "solver_type": "LinearSolversApplication.sparse_lu"
            },
            "model_import_settings": {
              "input_type": "mdpa",
              "input_filename": "mesh"
            },
            "material_im

## 7. The result: load-stepping

Each stage wrote to its own VTK folder. The tip deflection grows with the load — and doubles exactly, because each stage is a linear solve.

In [13]:
import meshio, numpy as np
def tip(folder):
    vtks = sorted((workdir / folder).glob("*.vtk"))
    m = meshio.read(str(vtks[-1])); pts = np.asarray(m.points)
    return float(np.asarray(m.point_data["DISPLACEMENT"])[np.argmax(pts[:, 0])][1])
t1, t2 = tip("vtk_load_step_1"), tip("vtk_load_step_2")
print(f"stage load_step_1 (1 MN/m): tip uy = {t1:.3e} m")
print(f"stage load_step_2 (2 MN/m): tip uy = {t2:.3e} m")
print(f"ratio (should be ~2.0): {t2 / t1:.3f}")

stage load_step_1 (1 MN/m): tip uy = -3.998e-04 m
stage load_step_2 (2 MN/m): tip uy = -7.996e-04 m
ratio (should be ~2.0): 2.000


## Cleanup\n\nClose the client session (which stops the server subprocess). The case directory is left on disk — open its `vtk_output/` in ParaView to inspect the raw results yourself.

In [14]:
_stop_event.set()      # wake the lifecycle task up ...
await _server_task     # ... and let it exit its own AsyncExitStack from the task that opened it
print("session closed, server subprocess terminated")
print("case directory left on disk:", workdir)

session closed, server subprocess terminated
case directory left on disk: /tmp/kratos-mcp-multistage-en_d2ngz
